In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "AI_RAG_LEGAL"

# 1. Clone/pull repo (update code hien co tren Kaggle)
GIT_URL = os.environ.get("GIT_REPO_URL", "https://github.com/noskaiser2310/AI_RAG_LEGAL.git")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "master", GIT_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "master"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/master"], check=True)

# 2. sys.path trick: bien Kaggle thanh moi truong local
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["HF_HOME"] = str(WORK / "hf_cache")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"
# JAX/TF ngu convert vao GPU 75% VRAM -> de CPU de tranh CUDA OOM
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
print("Repo:", REPO)
print("Python:", sys.executable)


In [ ]:
# 3. Loc data tu Kaggle Input (duong dan tuong minh + fallback tim de quy)
DATA_DIR = REPO / "data"
DATA_DIR.mkdir(exist_ok=True)

input_root = Path("/kaggle/input")
SLUG = "ai-rag-legal-assets"
src_pkg = None
for p in input_root.rglob(SLUG):
    if p.is_dir():
        src_pkg = p
        break
if src_pkg is None:
    raise RuntimeError(f"Chua tim thay dataset {SLUG} trong /kaggle/input")
print("Dataset path:", src_pkg)

# Duong dan da xac nhan tu cau truc dataset:
#   <root>/corpus/data/processed/corpus.jsonl
#   <root>/indexes/data/indexes/dense.index + sparse/
corpus_dir = src_pkg / "corpus" / "data" / "processed"
indexes_dir = src_pkg / "indexes" / "data" / "indexes"

# Fallback: tim de quy neu duong dan tren khong khop
if not (corpus_dir / "corpus.jsonl").exists():
    cand = list(src_pkg.rglob("corpus.jsonl"))
    corpus_dir = cand[0].parent if cand else corpus_dir
if not (indexes_dir / "dense.index").exists():
    cand = list(src_pkg.rglob("dense.index"))
    indexes_dir = cand[0].parent if cand else indexes_dir

print("corpus.jsonl src:", corpus_dir / "corpus.jsonl")
print("dense.index   src:", indexes_dir / "dense.index")
print("sparse        src:", indexes_dir / "sparse")

# DEBUG: liet ke xem cac thu muc trung gian co gi ne rope vao indexes
def debug_list(p: Path):
    print(f"  [{p}] parent_exists={p.parent.exists()} dir_exists={p.exists()}")
    if p.exists():
        items = sorted(x.name for x in p.iterdir())
        print(f"     contains: {items[:30]}")

for anc in [src_pkg, src_pkg / "indexes", src_pkg / "indexes" / "data", indexes_dir]:
    debug_list(anc)

# DEBUG chi tiet: trang thai tung entry ben trong indexes_dir
import os
for x in indexes_dir.iterdir():
    print(f"  ENTRY {x.name}: exists={x.exists()} is_file={x.is_file()} is_dir={x.is_dir()} is_symlink={x.is_symlink()} lexists={os.path.lexists(x)}")
    if x.is_symlink():
        print(f"     realpath={os.path.realpath(x)} -> target={os.readlink(x)}")
    if x.is_file():
        print(f"     size={x.stat().st_size}")

def symlink_or_copy(src: Path, dst: Path):
    if os.path.lexists(dst):
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        if src.is_dir():
            dst.symlink_to(src, target_is_directory=True)
        else:
            dst.symlink_to(src)
        print(f"  symlink {dst} -> {src}")
    except OSError:
        print(f"  Symlink fail, copying {src.name}...")
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)

# Link toan bo thu muc indexes (chua dense.index + sparse/) mot lan
dst_idx = DATA_DIR / "indexes"
# Neu data/indexes khong phai symlink thi xoa het (du lieu cu trong working la tam, co lai tu input)
if dst_idx.exists() and not dst_idx.is_symlink():
    print(f"  removing leftover {dst_idx}")
    shutil.rmtree(dst_idx)
if not os.path.lexists(dst_idx / "dense.index"):
    _ = os.stat(indexes_dir / "dense.index")  # touch de chac chan mount da materialize
    symlink_or_copy(indexes_dir, dst_idx)
if (corpus_dir / "corpus.jsonl").exists():
    symlink_or_copy(corpus_dir / "corpus.jsonl", DATA_DIR / "processed" / "corpus.jsonl")

idx = DATA_DIR / "indexes" / "dense.index"
sparse = DATA_DIR / "indexes" / "sparse"
corpus = DATA_DIR / "processed" / "corpus.jsonl"
print("dense.index:", os.path.lexists(idx))
print("sparse:", os.path.lexists(sparse))
print("corpus.jsonl:", os.path.lexists(corpus))

In [ ]:
# 3b. Symlink HF cache tu dataset ai-rag-legal-models vao hf_cache (Khong copy du lieu)
# Symlink chi la con tro ~0KB, mo tu /kaggle/input (read-only) truc tiep.
HF_HOME = Path(os.environ["HF_HOME"])
(HF_HOME / "hub").mkdir(parents=True, exist_ok=True)

MODEL_SLUG = "ai-rag-legal-models"
m_root = None
for p in input_root.rglob(MODEL_SLUG):
    if p.is_dir():
        m_root = p
        break
if m_root is None and (input_root / "datasets").exists():
    for owner in (input_root / "datasets").iterdir():
        cand = owner / MODEL_SLUG
        if cand.exists():
            m_root = cand
            break
if m_root is None:
    print("WARN: khong thay dataset ai-rag-legal-models - HF se tai lai model")
else:
    print("Models dataset:", m_root)
    hub_src = m_root / "hub" if (m_root / "hub").exists() else m_root
    for entry in hub_src.iterdir():
        name = entry.name
        if not (name.startswith("models--") or name.startswith("datasets--")):
            continue
        dst = HF_HOME / "hub" / name
        if not (dst.exists() or dst.is_symlink()):
            try:
                dst.symlink_to(entry, target_is_directory=True)
                print(f"  symlink {name} (0 copy)")
            except OSError as e:
                raise RuntimeError(
                    f"Khong the symlink {entry} -> {dst}: {e}. "
                    "Dung model tren /kaggle/input read-only - can symlink hoac thiet lap dossier."
                ) from e
    print("HF cache symlink OK - load truc tiep tu /kaggle/input")


In [ ]:
# 4. Tao .env cho local: device cuda, model embedding/reranker open-source
(REPO / ".env").write_text("\n".join([
    "EMBEDDING_MODEL=mainguyen9/vietlegal-harrier-0.6b",
    "EMBEDDING_DIM=1024",
    "RERANKER_MODEL=AITeamVN/Vietnamese_Reranker",
    "DEVICE=cpu",
    "EMBEDDING_DEVICE=cpu",
    "RERANKER_DEVICE=cpu",
    "RETRIEVAL_TOP_K=500",
    "RERANK_TOP_K=50",
    "FINAL_TOP_K=20",
    "SCORE_THRESHOLD=0.7",
    "MAX_TOKENS=2048",
    "TEMPERATURE=0.1",
    "DATA_DIR=data",
    "INDEX_DIR=data/indexes",
]), encoding="utf-8")
print(".env created")

# Local model mac dinh": "Qwen/Qwen3.5-4B",
# P100/T4: 4B tren 4-bit chay OK. Neu muon hon nua chon Qwen3-8B fp16/x4bit
LLM_MODEL = "Qwen/Qwen3.5-4B"


In [ ]:
# 5. Cai dependencies (torch/cuda co san tren Kaggle)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "transformers", "accelerate", "datasets", "faiss-cpu", "bm25s",
    "scikit-learn", "scipy", "tqdm", "pydantic-settings", "bitsandbytes",
], check=True)
print("Dependencies OK")

## 6. Chạy benchmark — model LOCAL

**Test nhanh 3 query trước** (không có script riêng; dùng batch idx 0, batch-size 5):

```bash
python -m evaluate.evaluate_vmteb_batch --batch-idx 0 --batch-size 5 --workers 1 --llm hf --hf-model Qwen/Qwen3.5-4B
```

Lưu ý model local: workers nên = 1 (1 GPU, suy luận tuần tự). Batch đầy đủ 50 query/ batch:

```bash
python -m evaluate.evaluate_vmteb_batch --batch-idx 0 --batch-size 50 --workers 1 --llm hf
```

**Toàn bộ 13 batch** (620 query, model local chậm hơn Gemini — ước tính 6-10/s

**Quan trọng:** kết quả được LƯU TỪNG QUERY ngay khi xong vào `data/results/vmteb_batch{N}_partial.jsonl`.
Batch nào đã có metrics file sẽ được SKIP khi chạy lại (dùng `--force` để chạy lại).
Session chết giữa chừng: chạy lại lệnh trên, script tự chạy tiếp phần còn thiếu, không mất dữ liệu.

```bash
python evaluate/run_all_batches.py --start 0 --end 12 --llm hf --hf-model Qwen/Qwen3.5-4B
```

**Resume batch lỗi**:

```bash
python -m evaluate.evaluate_vmteb_resume --batch-idx N --batch-size 50 --workers 1 --llm hf
```

Kết quả: `data/results/vmteb_batch{N}_metrics.json` (+ _detail.json, _trace.json)


In [ ]:
# Test 5 query dau.
ret = subprocess.run([sys.executable, "-m", "evaluate.evaluate_vmteb_batch",
    "--batch-idx", "0", "--batch-size", "5", "--workers", "1",
    "--llm", "hf", "--hf-model", LLM_MODEL], cwd=REPO)
print("Exit:", ret.returncode)

In [ ]:
# Full benchmark (620 query). Model local nen co the chia nhieu session:
# lan 1: --start 0 --end 6; lan 2: --start 7 --end 12
ret = subprocess.run([sys.executable, "evaluate/run_all_batches.py", "--start", "0", "--end", "12",
    "--llm", "hf", "--hf-model", LLM_MODEL], cwd=REPO)
print("Exit:", ret.returncode)

In [ ]:
# Xem tong hop diem
import json
res_dir = DATA_DIR / "results"
res_dir.mkdir(parents=True, exist_ok=True)
rows = []
for f in sorted(res_dir.glob("vmteb_batch*_metrics.json")):
    m = json.loads(f.read_text(encoding="utf-8"))
    rows.append((f.stem, m.get("num_queries"), m.get("num_errors"), m.get("macro_f2"), m.get("micro_f2"), m.get("mrr"), m.get("recall@5")))
import pandas as pd
df = pd.DataFrame(rows, columns=["batch", "n", "err", "macro_f2", "micro_f2", "mrr", "recall@5"])
display(df)
print(df[['macro_f2','micro_f2','mrr','recall@5']].mean())
df.to_csv("data/results/vmteb_summary.csv", index=False)
print("Saved: data/results/vmteb_summary.csv")


In [ ]:
# 9. Dong goi ket qua -> zip (chay sau moi session, tai zip ve may)
# Kaggle xoa /kaggle/working moi session -> phai tai zip ve truoc khi tat session!
import subprocess
ret = subprocess.run([sys.executable, "evaluate/package_results.py",
    "--out", str(REPO / "data" / "results")], cwd=REPO)
print("Exit:", ret.returncode)
# Sau khi chay: open file explorer ben trai -> mo data/results -> download file vmteb_results_*.zip
